# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Course:** LMA — Assignment 1

**Goal:** build two retrieval pipelines over *the same* chunked corpus and contrast them:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranked) |
|---|---|---|
| Retrieve | Bi-encoder, Top 10 | Bi-encoder, Top 10 (identical) |
| Select | Top 3 as-is | Cross-encoder re-sorts, Top 3 |
| Generate | LLM -> **Answer 1** | Same prompt -> **Answer 2** |

**Document:** _TODO: name + source URL of the PDF you chose_

---
## 1. Setup & Environment

Local-only stack: bi-encoder and cross-encoder from `sentence-transformers`, generation via Ollama.

In [ ]:
import hashlib
import json
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# pdfminer.six rather than pypdf -- see the note in section 2 for why.
from pdfminer.high_level import extract_pages
from pdfminer.layout import LAParams, LTChar, LTTextContainer, LTTextLineHorizontal

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import ollama

In [ ]:
# ---- Configuration: single source of truth for the whole notebook ----

# EU Regulation 2016/679 (GDPR), official EUR-Lex PDF, 88 pages, 99 Articles.
PDF_PATH = Path("data/GDPR.pdf")

# Bi-encoder: independent embeddings, used to build the vector index
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Cross-encoder: joint (query, chunk) scoring, used only to re-rank
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Generator (local, via Ollama)
LLM_MODEL = "qwen2.5:7b-instruct"
LLM_MODEL_FAST = "qwen2.5:3b-instruct"   # swap in while iterating; 7b for final runs

# Chunking
CHUNK_SIZE = 800      # characters
CHUNK_OVERLAP = 150

# Prefix each chunk with its own heading ("Article 6 - Lawfulness of processing").
# Keeps a chunk self-contained once it is torn out of the document, and lets the LLM
# cite properly. It also changes what the bi-encoder sees, so it is an experimental
# knob: flipping it is a fair thing to measure.
PREPEND_LABEL = True

# Retrieval depths mandated by the assignment
TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_N_CONTEXT = 3     # chunks actually shown to the LLM

RANDOM_SEED = 42

# Vector store. Persisted so the index survives a kernel restart; the fingerprint in
# cell 11 guards against silently serving stale vectors after a config change.
# (Embedding 652 chunks takes ~1s on MPS, so it is re-run each time regardless.)
CHROMA_PATH = ".chroma"
COLLECTION_NAME = "gdpr"

# -- Document structure ----------------------------------------------------
# Articles begin at "CHAPTER I"; everything before is preamble + 173 recitals.
# Recitals are NON-BINDING paraphrases of the Articles and are ~45% of the text,
# so tag every chunk with which side it came from -- you need that distinction
# to explain retrieval failures later.
ARTICLES_START_MARKER = "CHAPTER I"

In [ ]:
def check_environment() -> None:
    """Sanity-check that Ollama is up, LLM_MODEL is pulled, and MPS/CPU device is known.

    TODO:
      - ollama.list() -> confirm LLM_MODEL is present
      - print torch device that sentence-transformers will use (mps on Apple Silicon)
      - print library versions so the report can state the exact setup
    """
    raise NotImplementedError


check_environment()

---
## 2. Load & Chunk the Document

Both pipelines must share **the exact same corpus**, so this happens once, here.

### Why pdfminer and not pypdf

This PDF is letter-spaced, and `pypdf` reads those kerning gaps as word breaks. Its output
contains `Ar ticle` **507 times** (and `inf or mation` 163x, `pur poses` 129x, ...) while the
correct spellings never appear at all. Those are not cosmetic: corrupted tokens embed to
meaningless vectors, so every similarity score in both pipelines would be computed over
damaged text and the comparison would measure an extraction bug rather than the two retrieval
architectures.

The first instinct is to repair the text afterwards with a dictionary or substitution table.
That is the wrong layer. The spaces are not in the PDF -- they are *inferred* by the extractor
from glyph positions, so the fix belongs in extraction. `pdfminer.six` uses a different spacing
threshold and produces **zero** broken words on this document with default settings.

### Why line geometry and not `extract_text()`

pdfminer's convenience function emits whole *text boxes* in its own order, and the numbered
paragraph markers (`1.`, `2.`) live in the left margin as separate boxes -- which it emits
*after* the paragraph body, splitting 280 paragraphs mid-sentence.

The coordinates give this away exactly. On page 40:

```
x0=67.4  y1=766.0   "2."
x0=89.0  y1=766.0   "The controller shall facilitate the exercise of ..."
```

Same `y1`: the marker and its paragraph's first line are **on the same visual row**, marker to
the left. So reconstructing the page from line geometry and sorting by `(-y, x)` restores the
true reading order with no heuristics. Grouping lines that share a row is what reattaches each
marker to its own paragraph.

### Two more extraction traps in this PDF

- **Hyphenation across line breaks.** Justified text hyphenates with a soft hyphen (U+00AD).
  Stripping the hyphen and joining lines with a space produces `adminis trative`,
  `propor tionate`, `identifi cation` -- the same class of broken token, reintroduced at a
  later stage. Hyphenated lines must be joined with *no* space.
- **Letter-spaced SECTION headings.** These are set with wide tracking for display, and no
  spacing heuristic can recover them: `S ect i on 4 Dat a p ro te ctio n o ff ice r`. They are
  detected by the share of 1-2 character tokens (real prose stays under 0.6, every tracked
  heading is 0.73+), `Section N` is recovered from the number, and the mangled title is
  dropped. Some titles sit at only ~0.60 because they open with real words, so those are
  caught by position instead -- a tracked title always follows its `Section N` line.

Verify both with the open-ended sweep in the QA cell rather than a fixed list of known-bad
forms. A fixed list only finds the damage you already thought of: the sweep is what surfaced
the hyphenation class in the first place.

### Remaining decisions

- **Footnotes** (8.4-8.5pt citations in the bottom margin) are dropped: they interrupt body
  text and add nothing retrievable. Note that `CHAPTER` headings are *also* 8.5pt, so the
  filter has to test size **and** vertical position -- size alone deletes the chapter headings.
- **Chunk size is not a neutral knob.** At 800 chars an "except when..." clause may land in the
  same chunk as the rule it modifies, which *erases* the failure mode you want to study; much
  smaller and the exception loses the article number that anchors it. Record the value you used
  and treat its effect as a finding, not a nuisance.
- **Chunks never cross a section boundary** (one Article or one Recital), so every chunk's
  `label` is truthful and citable. The cost is some short chunks where a Recital is brief.
- **`PREPEND_LABEL`** puts `Article 6 - Lawfulness of processing` at the top of each chunk. It
  makes a chunk self-contained once torn out of the document and lets the LLM cite properly,
  but it also changes what the bi-encoder sees -- flipping it is a fair thing to measure.
- **Overlap duplicates text.** At 150/800 the corpus inflates ~1.15x and a passage can appear
  in two adjacent chunks, so a Top-10 list can spend two slots on near-duplicates. Worth a
  sentence in the report when you discuss what the baseline retrieved.

In [ ]:
# Page furniture, repeated on every page in varying order.
PAGE_FURNITURE_RE = re.compile(
    r"^(?:EN|L\s*119/\d+|Official\s+Journal\s+of\s+the\s+European\s+Union|\d{1,2}\.\d{1,2}\.\d{4})$"
)
SECTION_RE = re.compile(r"^Section(\d+)$", re.I)

ROW_TOL = 2.0           # points; lines within this vertical distance share a visual row
BODY_FONT_MIN = 9.0     # body text is 9.6pt; footnote citations are 8.4-8.5pt
FOOTNOTE_MAX_Y = 130.0  # ...but CHAPTER headings are 8.5pt too, so require the bottom margin
TRACKED_DENSITY = 0.7   # fraction of 1-2 char tokens that marks a letter-spaced heading
SOFT_HYPHEN = "­"


def _font_size(line) -> float:
    """Mean glyph size of a text line, used to tell body text from footnotes."""
    sizes = [c.size for c in line if isinstance(c, LTChar)]
    return sum(sizes) / len(sizes) if sizes else 0.0


def _tracked_density(text: str) -> float:
    """Fraction of very short tokens -- high in display headings set with wide tracking.

    The SECTION headings in this PDF are letter-spaced for display, which no spacing
    heuristic can undo: "S ect i on 4 Dat a p ro te ctio n o ff ice r". Measured across
    the document, real prose stays below 0.6 and every tracked heading is 0.73+.
    """
    toks = text.split()
    if len(toks) < 4:
        return 0.0
    return sum(1 for t in toks if len(t) <= 2) / len(toks)


def _normalise(s: str) -> str:
    """Strip typographic noise. Soft hyphens are kept for the line-join pass below."""
    s = s.replace("‘", "'").replace("’", "'")   # curly quotes
    s = s.replace("‑", "-").replace("—", "--")  # non-breaking hyphen, em dash
    return re.sub(r"[ \t]+", " ", s).strip()


def _join_hyphenated(lines: list[str]) -> list[str]:
    """Rejoin words split across a line break by hyphenation.

    A line ending in a soft hyphen continues in the next line, so it must be joined with
    NO space: dropping the hyphen and joining with a space yields "adminis trative",
    which is exactly the kind of broken token this pipeline exists to avoid.
    """
    out: list[str] = []
    for line in lines:
        if out and out[-1].endswith(SOFT_HYPHEN) and line:
            out[-1] = out[-1][:-1] + line
        else:
            out.append(line)
    return [ln.replace(SOFT_HYPHEN, "") for ln in out]


def load_pdf_text(pdf_path: Path, drop_footnotes: bool = True) -> list[dict]:
    """Extract the PDF one record per page, in true reading order.

    Rebuilt from line geometry rather than pdfminer's extract_text(), which emits the
    left-margin paragraph markers after the paragraph body. See the notes above.

    Returns: [{"page", "text", "n_footnotes", "n_tracked"}, ...]   # page is 1-based
    """
    pages = []
    for pno, layout in enumerate(extract_pages(str(pdf_path), laparams=LAParams()), start=1):
        rows, n_foot, n_tracked = [], 0, 0
        for el in layout:
            if not isinstance(el, LTTextContainer):
                continue
            for ln in el:
                if not isinstance(ln, LTTextLineHorizontal):
                    continue
                txt = _normalise(ln.get_text())
                if not txt or PAGE_FURNITURE_RE.match(txt):
                    continue
                if drop_footnotes and _font_size(ln) < BODY_FONT_MIN and ln.y1 < FOOTNOTE_MAX_Y:
                    n_foot += 1
                    continue
                if _tracked_density(txt) > TRACKED_DENSITY:
                    # Recover "Section N" where possible; the tracked title is unrecoverable.
                    m = SECTION_RE.match(re.sub(r"\s+", "", txt))
                    n_tracked += 1
                    if not m:
                        continue
                    txt = f"Section {m.group(1)}"
                rows.append((ln.y1, ln.x0, txt))

        # Group lines sharing a visual row, so a margin marker joins its own first line.
        buckets: dict[int, list[tuple[float, str]]] = {}
        for y1, x0, txt in rows:
            buckets.setdefault(round(y1 / ROW_TOL), []).append((x0, txt))

        # True reading order: top to bottom, then left to right within a row.
        out, prev_key = [], None
        for key in sorted(buckets, reverse=True):
            line = " ".join(txt for _, txt in sorted(buckets[key]))
            # A tracked SECTION title always follows its "Section N" line, and some are
            # only ~0.6 dense because they open with real words ("Right to object and
            # au to ma te d ..."). Catch those by position rather than by lowering the
            # threshold into the range of ordinary prose.
            if (out and re.fullmatch(r"Section \d+", out[-1])
                    and _tracked_density(line) > 0.45):
                n_tracked += 1
                continue
            if prev_key is not None and (prev_key - key) * ROW_TOL > 14:
                out.append("")                       # larger vertical gap => new paragraph
            out.append(line)
            prev_key = key

        text = re.sub(r"\n{3,}", "\n\n", "\n".join(_join_hyphenated(out))).strip()
        pages.append({"page": pno, "text": text, "n_footnotes": n_foot, "n_tracked": n_tracked})
    return pages


pages = load_pdf_text(PDF_PATH)
print(f"{len(pages)} pages, {sum(len(p['text']) for p in pages):,} chars, "
      f"{sum(p['n_footnotes'] for p in pages)} footnote lines dropped, "
      f"{sum(p['n_tracked'] for p in pages)} tracked heading lines handled")

In [ ]:
# ---- Extraction QA: prove the corpus is sound BEFORE embedding anything ----
#
# Embedding is the point of no return: once chunks are indexed, corrupted text is invisible
# and simply degrades retrieval quietly. So assert, don't eyeball.

_full = "\n".join(p["text"] for p in pages)
_flat = re.sub(r"\s+", " ", _full)

# 1. No kerning damage. The first group is what pypdf produced; the second is what
#    survived pdfminer until hyphenated line-joins were handled properly.
for _w in ["Ar ticle", "inf or mation", "pur poses", "r ights", "U nion", "car r ied", "fi rst",
           "adminis trative", "propor tionate", "investi gation", "identifi cation",
           "certifi cation", "pseudony misation", "communi cation", "internat ional",
           "indepen dently", "consoli dation", "physio logical", "jurisdic tional"]:
    assert _flat.count(_w) == 0, f"broken form still present: {_w!r}"

# 2. Nothing over-joined: real word boundaries survived.
for _w in ["ofthe", "tothe", "inthe", "andthe", "shallbe"]:
    assert _flat.count(_w) == 0, f"words glued together: {_w!r}"
for _w in ["of the", "shall be", "in the"]:
    assert _flat.count(_w) > 50, f"expected bigram missing: {_w!r}"

# 3. Paragraph markers sit with their paragraphs, not orphaned on their own line.
assert len(re.findall(r"\n\d{1,2}\.\n", _full)) == 0, "orphaned paragraph markers remain"

# 4. Passages this assignment depends on are present and contiguous.
for _w in ["Point (f) of the first subparagraph shall not apply",   # Art 6(1) carve-out
           "sexual orientation shall be prohibited",                # Art 9(1)
           "Paragraph 1 shall not apply if one of the following",   # Art 9(2)
           "fewer than 250 persons",                                # Art 30(5)
           "20 000 000"]:                                           # Art 83(5)
    assert _flat.count(_w) >= 1, f"key passage missing: {_w!r}"

# 5. Structure survived.
assert len(re.findall(r"(?:^|\n)Article \d+\s*\n", _full)) == 99, "expected 99 Article headings"
assert _flat.count(ARTICLES_START_MARKER) >= 1
assert all(p["text"].strip() for p in pages), "some page came out empty"

# 6. Open-ended sweep for damage no fixed list would catch: a short token adjacent to
#    another where the merge is a dictionary word but the parts are not both words.
#    This is a REPORT, not an assert -- a handful of false positives are expected
#    ("be held" -> "beheld", "a box" -> "abox"), so read the list rather than trusting a
#    threshold. If genuine damage appears here, fix it at the extraction layer.
try:
    _vocab = {w.strip().lower() for w in open("/usr/share/dict/words")}
    _vocab |= {"rights", "rules", "purposes", "fines", "referred", "proportionate",
               "organisation", "pseudonymisation", "recognised", "safeguards", "profiling"}
    _toks = _flat.split(" ")
    _sus: dict[str, int] = {}
    for _a, _b in zip(_toks, _toks[1:]):
        _a2, _b2 = re.sub(r"[^A-Za-z]", "", _a), re.sub(r"[^A-Za-z]", "", _b)
        if not _a2 or not _b2 or len(_a2) > 8 or len(_b2) > 10:
            continue
        if ((_a2 + _b2).lower() in _vocab
                and not (_a2.lower() in _vocab and _b2.lower() in _vocab)):
            _sus[f"{_a2} {_b2}"] = _sus.get(f"{_a2} {_b2}", 0) + 1
    print(f"residual broken-word candidates: {sum(_sus.values())} "
          f"({len(_sus)} distinct) -> {sorted(_sus, key=_sus.get, reverse=True)[:8]}")
except FileNotFoundError:
    print("no system dictionary; skipped the open-ended sweep")

_n_articles = len(re.findall(r"(?:^|\n)Article \d+\s*\n", _full))
_n_recitals = len(re.findall(r"(?:^|\n)\(\d{1,3}\) ", _full))
print("extraction QA passed")
print(f"  Article headings : {_n_articles}")
print(f"  recital markers  : {_n_recitals}")
print(f"  total characters : {len(_full):,}")

In [ ]:
# ---- Structure-aware chunking ------------------------------------------------
# GDPR has a very regular shape, which is worth exploiting:
#
#     CHAPTER I
#     General provisions
#     Article 1                 <- heading, alone on a line (99 of them)
#     Subject-matter and objectives   <- title, on the next line
#     1. This Regulation lays down ...
#     2. ...
#
# Chunks are packed to CHUNK_SIZE but never cross a section boundary (one Article or
# one Recital), so every chunk's label is truthful. That is what makes the rank-shift
# analysis in section 6 citable rather than anecdotal.

ARTICLE_RE = re.compile(r"^Article (\d+)$")
CHAPTER_RE = re.compile(r"^CHAPTER ([IVX]+)$")
RECITAL_RE = re.compile(r"^\((\d{1,3})\)\s+(.*)$", re.S)
SENT_END_RE = re.compile(r"(?<=[.;:])\s+")


@dataclass
class Chunk:
    """One indexed unit of the corpus."""
    id: str                      # stable, e.g. "c0042" -- cited in the rank tables
    text: str
    page: int
    kind: str                    # "recital" | "article" | "preamble"
    label: str | None = None     # "Article 6", "Recital 47"
    title: str | None = None     # "Lawfulness of processing"


def iter_paragraphs(pages: list[dict]) -> list[dict]:
    """Flatten pages into paragraphs, rejoining any split by a page break."""
    paras: list[dict] = []
    for p in pages:
        for block in p["text"].split("\n\n"):
            text = re.sub(r"\s+", " ", block).strip()
            if not text:
                continue
            # Continued from the previous page: the previous paragraph did not end a
            # sentence and this one does not open a new structural element.
            if paras:
                prev = paras[-1]["text"]
                starts_new = (ARTICLE_RE.match(text) or CHAPTER_RE.match(text)
                              or re.match(r"^\(?\d{1,3}[.)]\s", text)
                              or re.match(r"^\([a-z]\)\s", text)
                              or text[0].isupper())
                if not prev.endswith((".", ";", ":", "?", "!")) and not starts_new:
                    paras[-1]["text"] = f"{prev} {text}"
                    continue
            paras.append({"text": text, "page": p["page"]})
    return paras


def annotate_structure(paras: list[dict]) -> list[dict]:
    """Tag each paragraph with kind / label / title by walking the document in order."""
    out = []
    in_articles = False
    label = title = None
    kind = "preamble"
    expect_title = False

    for para in paras:
        text = para["text"]

        if CHAPTER_RE.match(text) or text == ARTICLES_START_MARKER:
            in_articles = True
            continue                                   # heading carries no content

        m = ARTICLE_RE.match(text)
        if m:
            in_articles, kind = True, "article"
            label, title, expect_title = f"Article {m.group(1)}", None, True
            continue
        if expect_title:                               # the line right after "Article N"
            title, expect_title = text, False
            continue

        if not in_articles:
            r = RECITAL_RE.match(text)
            if r:
                kind, label, title = "recital", f"Recital {r.group(1)}", None
                text = r.group(2).strip()              # drop the "(47) " marker
            elif label is None:
                kind = "preamble"

        out.append({**para, "text": text, "kind": kind, "label": label, "title": title})
    return out


def _split_words(text: str, limit: int) -> list[str]:
    """Last-resort split on word boundaries, for a 'sentence' longer than the budget."""
    cur, pieces = "", []
    for w in text.split(" "):
        if cur and len(cur) + 1 + len(w) > limit:
            pieces.append(cur)
            cur = w
        else:
            cur = f"{cur} {w}".strip()
    if cur:
        pieces.append(cur)
    return pieces


def _split_long(text: str, limit: int) -> list[str]:
    """Break an over-long paragraph on sentence boundaries, words as a fallback."""
    cur, pieces = "", []
    for s in SENT_END_RE.split(text):
        if len(s) > limit:                             # a single clause exceeds the budget
            if cur:
                pieces.append(cur)
                cur = ""
            pieces.extend(_split_words(s, limit))
            continue
        if cur and len(cur) + 1 + len(s) > limit:
            pieces.append(cur)
            cur = s
        else:
            cur = f"{cur} {s}".strip()
    if cur:
        pieces.append(cur)
    return pieces


def _tail(text: str, n: int) -> str:
    """Trailing ~n chars for the overlap, never starting mid-word.

    Prefer a sentence boundary, fall back to a word boundary. Cutting blindly at n
    characters yields chunks beginning "ssing is necessary for ...", which is both
    unreadable and poor input to the embedding model.
    """
    if n <= 0:
        return ""
    if len(text) <= n:
        return text
    tail = text[-n:]
    parts = SENT_END_RE.split(tail, maxsplit=1)
    if len(parts) > 1 and parts[-1].strip():
        return parts[-1].strip()
    space = tail.find(" ")
    return tail[space + 1:].strip() if space != -1 else tail.strip()


def build_chunks(paras: list[dict], chunk_size: int, overlap: int) -> list[Chunk]:
    """Pack annotated paragraphs into overlapping chunks that never cross a section."""
    chunks: list[Chunk] = []
    buf: list[dict] = []

    def flush() -> None:
        if not buf:
            return
        head = buf[0]
        body = " ".join(p["text"] for p in buf)
        text = body
        if PREPEND_LABEL and head["label"]:
            hdr = head["label"] + (f" - {head['title']}" if head["title"] else "")
            text = f"{hdr}\n\n{body}"
        chunks.append(Chunk(id=f"c{len(chunks):04d}", text=text, page=head["page"],
                            kind=head["kind"], label=head["label"], title=head["title"]))
        buf.clear()

    prev_key = None
    for para in paras:
        key = (para["kind"], para["label"])
        if key != prev_key:                            # new Article / Recital: start fresh
            flush()
            prev_key = key

        pieces = (_split_long(para["text"], chunk_size)
                  if len(para["text"]) > chunk_size else [para["text"]])
        for piece in pieces:
            cur_len = sum(len(p["text"]) + 1 for p in buf)
            if buf and cur_len + len(piece) > chunk_size:
                carried = _tail(" ".join(p["text"] for p in buf), overlap)
                flush()
                if carried:
                    buf.append({**para, "text": carried})
            buf.append({**para, "text": piece})
    flush()
    return chunks


def locate(chunks: list[Chunk], phrase: str) -> list[Chunk]:
    """Every chunk containing `phrase` -- how you find gold chunk ids by hand."""
    needle = re.compile(re.escape(re.sub(r"\s+", " ", phrase)), re.I)
    return [c for c in chunks if needle.search(re.sub(r"\s+", " ", c.text))]

In [ ]:
# ---- Build the corpus and interrogate it before indexing ----

paragraphs = iter_paragraphs(pages)
annotated = annotate_structure(paragraphs)
chunks = build_chunks(annotated, CHUNK_SIZE, CHUNK_OVERLAP)

_lens = sorted(len(c.text) for c in chunks)
print(f"{len(paragraphs)} paragraphs -> {len(chunks)} chunks")
print(f"chunk chars: min={_lens[0]} median={_lens[len(_lens)//2]} max={_lens[-1]}")
print(pd.Series([c.kind for c in chunks]).value_counts().to_string())

# Structure must have survived chunking: GDPR has exactly 99 Articles and 173 Recitals.
assert len({c.label for c in chunks if c.kind == "article"}) == 99
assert len({c.label for c in chunks if c.kind == "recital"}) == 173
assert all(c.text.strip() for c in chunks), "a chunk came out empty"
assert all(c.label or c.kind == "preamble" for c in chunks), "unlabelled non-preamble chunk"
print("\nstructure checks passed")

# ---- Ground truth: where the passages this assignment needs actually landed ----
# Record these ids now; section 6 cites them when reporting rank shifts.
GOLD = {
    "art6_public_authority": "Point (f) of the first subparagraph shall not apply",
    "art9_prohibition":      "sexual orientation shall be prohibited",
    "art9_exceptions":       "Paragraph 1 shall not apply if one of the following",
    "art30_250_persons":     "fewer than 250 persons",
    "art83_fine_amount":     "20 000 000",
    "art84_penalties":       "rules on other penalties applicable to infringements",
    "recital47_legit_int":   "Given that it is for the legislator to provide by law",
}

_rows = []
for name, phrase in GOLD.items():
    hits = locate(chunks, phrase)
    assert hits, f"gold passage not found in any chunk: {name}"
    _rows.append({"passage": name, "chunk_ids": ", ".join(c.id for c in hits),
                  "label": hits[0].label, "kind": hits[0].kind, "page": hits[0].page})
gold_map = pd.DataFrame(_rows)
display(gold_map)

# Note: CHUNK_OVERLAP means a passage can appear in two adjacent chunks. That is fine,
# but remember it when reading Top-10 lists -- two near-duplicate hits spend two slots.

---
## 3. Index the Corpus (Bi-Encoder)

The bi-encoder embeds each chunk **independently** and offline. The query is embedded the same way at
search time, and relevance is a similarity between two fixed vectors — the model never sees the query
and the chunk together.

In [ ]:
def pick_device() -> str:
    """mps on Apple Silicon, cpu elsewhere."""
    return "mps" if torch.backends.mps.is_available() else "cpu"


def corpus_fingerprint(chunks: list[Chunk], embed_model_name: str) -> str:
    """Identify this exact corpus + model, so a cached index is only reused when valid.

    Without this, changing CHUNK_SIZE or the embedding model would silently keep serving
    the old vectors and every downstream number would be wrong.
    """
    h = hashlib.sha256()
    h.update(f"{embed_model_name}|{CHUNK_SIZE}|{CHUNK_OVERLAP}|{PREPEND_LABEL}|{len(chunks)}".encode())
    for c in chunks:
        h.update(c.id.encode())
        h.update(c.text.encode())
    return h.hexdigest()[:16]


def embed_texts(model: SentenceTransformer, texts: list[str], batch_size: int = 64) -> np.ndarray:
    """Encode texts into an (n, dim) float32 matrix, L2-normalised.

    Normalising makes inner product identical to cosine similarity, so the vector store's
    distance and any manual dot product agree.
    """
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True).astype(np.float32)


def build_vector_store(chunks: list[Chunk], embeddings: np.ndarray, fingerprint: str):
    """Create, or reuse, a persistent Chroma collection holding the chunk vectors.

    Returns (collection, cache_hit).
    """
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    if COLLECTION_NAME in [c.name for c in client.list_collections()]:
        col = client.get_collection(COLLECTION_NAME)
        if col.metadata and col.metadata.get("fingerprint") == fingerprint:
            return col, True
        client.delete_collection(COLLECTION_NAME)      # corpus changed; rebuild

    col = client.create_collection(
        COLLECTION_NAME,
        metadata={"hnsw:space": "cosine", "fingerprint": fingerprint},
    )
    for i in range(0, len(chunks), 256):               # Chroma caps batch size
        batch = chunks[i:i + 256]
        col.add(
            ids=[c.id for c in batch],
            embeddings=embeddings[i:i + 256].tolist(),
            documents=[c.text for c in batch],
            metadatas=[{"page": c.page, "kind": c.kind,
                        "label": c.label or "", "title": c.title or ""} for c in batch],
        )
    return col, False

In [ ]:
# ---- Load the bi-encoder, embed the corpus, build the collection ----

DEVICE = pick_device()
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"bi-encoder on {DEVICE}, dim={embedder.get_sentence_embedding_dimension()}")

_fp = corpus_fingerprint(chunks, EMBED_MODEL)

_t0 = time.perf_counter()
embeddings = embed_texts(embedder, [c.text for c in chunks])
INDEX_EMBED_S = time.perf_counter() - _t0

collection, _cache_hit = build_vector_store(chunks, embeddings, _fp)

# Normalisation is what makes cosine == inner product; check it rather than assume it.
_norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(_norms, 1.0, atol=1e-3), f"vectors not normalised: {_norms.min()}..{_norms.max()}"
assert collection.count() == len(chunks)

print(f"embedded {embeddings.shape} in {INDEX_EMBED_S:.1f}s "
      f"({len(chunks)/INDEX_EMBED_S:.0f} chunks/s)")
print(f"collection '{COLLECTION_NAME}': {collection.count()} vectors, cache_hit={_cache_hit}")
print(f"\nONE-OFF INDEXING COST: {INDEX_EMBED_S:.1f}s for {len(chunks)} chunks")
print("Report this separately from per-query latency in section 7 -- it is paid once,")
print("whereas the cross-encoder cost in Pipeline 2 is paid on every single query.")

---
## 4. Pipeline 1 — Baseline Retrieval

Retrieve Top 10 by vector similarity, hand the **Top 3** straight to the LLM -> **Answer 1**.

```mermaid
flowchart TD
    subgraph OFFLINE["INDEXING — paid once, ~1.0 s total"]
        direction TB
        D["GDPR PDF<br/>88 pages"] --> E["pdfminer.six<br/>geometry-based extraction"]
        E --> F["652 chunks<br/>800 chars / 150 overlap<br/>never crossing a section"]
        F --> G["Bi-encoder<br/>all-MiniLM-L6-v2<br/>652 forward passes"]
        G --> H[("Chroma index<br/>652 x 384-d vectors<br/>cosine / HNSW")]
    end

    Q(["User query"]) --> QE["Bi-encoder — SAME model<br/>1 forward pass<br/>p50 6.6 ms"]
    QE --> V["Query vector<br/>384-d, L2-normalised"]
    V --> S["Cosine similarity search<br/>p50 1.2 ms"]
    H -. "precomputed vectors<br/>reused every query" .-> S
    S --> T10["Top-10 candidates<br/>ordered by cosine"]
    T10 --> T3["Top-3 — taken UNCHANGED"]
    T10 -. "ranks 4-10 discarded<br/>unexamined" .-> X["(dropped)"]
    T3 --> PR["PROMPT_TEMPLATE<br/>context + question"]
    PR --> L["qwen2.5:7b-instruct<br/>temp 0, seed 42<br/>p50 3.25 s"]
    L --> A1(["ANSWER 1"])

    classDef store fill:#e8eaf6,stroke:#3f51b5,stroke-width:1px
    classDef out fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    classDef drop fill:#fafafa,stroke:#bdbdbd,stroke-dasharray:3 3,color:#9e9e9e
    class H store
    class A1 out
    class X drop
```

The orange-free path above is the whole of Pipeline 1: the cheap stage both *selects* and
*orders*. Ranks 4-10 are retrieved and then thrown away without ever being examined more
closely, which is exactly the gap Pipeline 2 fills.

In [ ]:
@dataclass
class Candidate:
    """A retrieved chunk with whatever scores it has accumulated."""
    chunk_id: str
    text: str
    page: int
    kind: str                  # carried through so you can spot "answered from a recital"
    label: str | None
    dense_rank: int            # 1-based rank from the bi-encoder
    dense_score: float
    cross_score: float | None = None
    cross_rank: int | None = None


def retrieve_baseline(query: str, collection, embed_model, k: int = TOP_K_RETRIEVE):
    """Embed the query, return (Top-k candidates, stage timings).

    The collection is configured for cosine *distance*; dense_score is reported as
    similarity (1 - distance) so that higher always means more relevant.

    Timings are returned per stage because query embedding and vector search have very
    different costs, and section 7 needs them apart.
    """
    t0 = time.perf_counter()
    qvec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    t1 = time.perf_counter()
    res = collection.query(query_embeddings=qvec.tolist(), n_results=k,
                           include=["documents", "metadatas", "distances"])
    t2 = time.perf_counter()

    cands = []
    for rank, (cid, doc, meta, dist) in enumerate(zip(
            res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]), start=1):
        cands.append(Candidate(chunk_id=cid, text=doc, page=int(meta["page"]),
                               kind=meta["kind"], label=meta["label"] or None,
                               dense_rank=rank, dense_score=1.0 - float(dist)))
    return cands, {"embed_query_s": t1 - t0, "search_s": t2 - t1}


def show_candidates(candidates: list[Candidate], width: int = 70) -> pd.DataFrame:
    """Compact view: rank | id | kind | label | scores | preview.

    You will stare at this more than any other output in the assignment -- it is how you
    find failure queries in section 6.
    """
    rows = []
    for c in candidates:
        body = c.text.split("\n\n")[-1]            # strip the prepended heading
        rows.append({"rank": c.dense_rank, "xrank": c.cross_rank, "id": c.chunk_id,
                     "kind": c.kind, "label": c.label, "pg": c.page,
                     "dense": round(c.dense_score, 4),
                     "cross": None if c.cross_score is None else round(c.cross_score, 3),
                     "preview": body[:width].replace("\n", " ")})
    return pd.DataFrame(rows)

In [ ]:
# ONE prompt, reused byte-for-byte by both pipelines. If the prompts differ, the
# comparison measures prompt wording instead of retrieval quality.
PROMPT_TEMPLATE = """\
You are answering a question about the EU General Data Protection Regulation (GDPR) \
using ONLY the numbered extracts below.

Rules:
1. Use only the extracts. Do not rely on any other knowledge of the GDPR.
2. If the extracts do not fully support an answer, begin that sentence with \
"INSUFFICIENT CONTEXT:" and say what is missing. Do not guess.
3. Cite the extract id in square brackets after each claim, e.g. [c0324].
4. If one extract states a general rule and another states an exception, a limit or a \
condition, state both.
5. Answer in at most 150 words.

Extracts:
{context}

Question: {question}

Answer:"""


def format_context(candidates: list[Candidate]) -> str:
    """Render the selected chunks into the {context} slot, labelled with id and page.

    The id is what the model cites, which is how you trace an answer back to the chunk
    that produced it -- essential for the Answer 1 vs Answer 2 comparison.
    """
    parts = []
    for c in candidates:
        head = f"[{c.chunk_id}] ({c.label or 'preamble'}, {c.kind}, page {c.page})"
        parts.append(f"{head}\n{c.text}")
    return "\n\n".join(parts)


def generate_answer(question: str, candidates: list[Candidate], model: str = LLM_MODEL) -> str:
    """Call the local LLM with the shared prompt template.

    temperature=0 and a fixed seed so that any difference between Answer 1 and Answer 2
    comes from the retrieved CONTEXT, not from sampling noise.
    """
    prompt = PROMPT_TEMPLATE.format(context=format_context(candidates), question=question)
    res = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}],
                      options={"temperature": 0, "seed": RANDOM_SEED, "num_ctx": 8192})
    return res["message"]["content"].strip()

In [ ]:
def run_pipeline_1(query: str, model: str = LLM_MODEL) -> dict:
    """Baseline: Top-10 dense retrieval -> Top-3 as-is -> Answer 1."""
    cands, timings = retrieve_baseline(query, collection, embedder, TOP_K_RETRIEVE)
    context = cands[:TOP_N_CONTEXT]

    t0 = time.perf_counter()
    answer = generate_answer(query, context, model=model)
    timings["generation_s"] = time.perf_counter() - t0
    timings["retrieval_s"] = timings["embed_query_s"] + timings["search_s"]

    return {"pipeline": "baseline", "query": query, "candidates": cands,
            "context_used": context, "answer": answer, "timings": timings}


# ---- Smoke test: a query whose answer can be checked by hand in the PDF ----
_q = "What is the maximum administrative fine under the GDPR?"
_out = run_pipeline_1(_q)
display(show_candidates(_out["candidates"]))
print({k: round(v, 3) for k, v in _out["timings"].items()})
print("\nANSWER 1:\n" + _out["answer"])

# Art 83(5) sets 20 000 000 EUR / 4 %; if that is missing, something upstream is wrong.
assert "20" in _out["answer"] and ("000 000" in _out["answer"] or "000,000" in _out["answer"]), \
    "baseline failed the smoke test -- check the index before going further"
print("\nsmoke test passed")

---
## 5. Pipeline 2 — Cross-Encoder Re-Ranking

Same Top-10 candidate set. Each `(query, chunk)` pair goes through the cross-encoder **jointly**, so
self-attention runs across query and chunk tokens together. Re-sort by that score, take the new Top 3,
and use the **identical prompt** -> **Answer 2**.

```mermaid
flowchart TD
    subgraph STAGE1["STAGE 1 — byte-for-byte identical to Pipeline 1"]
        direction TB
        Q(["User query"]) --> QE["Bi-encoder<br/>p50 6.6 ms"]
        QE --> S["Cosine search over the same<br/>652 precomputed vectors<br/>p50 1.2 ms"]
        S --> T10["Top-10 candidates<br/>SAME candidate set"]
    end

    T10 --> PAIR["Form 10 pairs:<br/>query + chunk concatenated<br/>into a SINGLE input"]
    PAIR --> CE["Cross-encoder<br/>ms-marco-MiniLM-L-6-v2<br/>joint self-attention across<br/>query and chunk tokens<br/>10 forward passes, p50 19.4 ms"]
    CE --> SC["10 relevance logits<br/>unbounded; comparable within<br/>a query, NOT across queries"]
    SC --> RS["Re-sort descending<br/>dense_rank kept, so the<br/>rank SHIFT stays visible"]
    RS --> T3["New Top-3<br/>may promote a chunk<br/>from rank 4-10"]
    T3 --> PR["PROMPT_TEMPLATE<br/>identical bytes, identical model"]
    PR --> L["qwen2.5:7b-instruct<br/>temp 0, seed 42"]
    L --> A2(["ANSWER 2"])

    NOTE["Nothing here can be precomputed:<br/>the score depends on the PAIR,<br/>so the full cost is paid per query"]
    NOTE -.-> CE

    classDef diff fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef out fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    classDef note fill:#fffde7,stroke:#f9a825,stroke-dasharray:4 3
    class PAIR,CE,SC,RS diff
    class A2 out
    class NOTE note
```

Highlighted boxes are the *only* difference from Pipeline 1. Everything before them (corpus,
index, candidate set) and after them (prompt, model, seed) is identical, so any change in the
answer is attributable to which 3 of the same 10 chunks were chosen.

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE)
print(f"cross-encoder on {DEVICE}: {CROSS_ENCODER_MODEL}")


def rerank(query: str, candidates: list[Candidate], cross_encoder: CrossEncoder):
    """Score every (query, chunk) pair jointly and re-sort descending.

    All TOP_K candidates are scored, not just the top few -- the point is to give chunks
    sitting at ranks 4-10 a chance to be promoted.

    dense_rank is left intact so the rank SHIFT stays visible; that shift is the headline
    result of this assignment.
    """
    pairs = [(query, c.text) for c in candidates]
    t0 = time.perf_counter()
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    elapsed = time.perf_counter() - t0

    for c, s in zip(candidates, scores):
        c.cross_score = float(s)
    ordered = sorted(candidates, key=lambda c: c.cross_score, reverse=True)
    for rank, c in enumerate(ordered, start=1):
        c.cross_rank = rank
    return ordered, {"rerank_s": elapsed, "n_pairs": len(pairs)}


def run_pipeline_2(query: str, model: str = LLM_MODEL) -> dict:
    """Re-ranked: Top-10 dense -> cross-encoder re-sort -> Top-3 -> Answer 2.

    Note the cross-encoder scores are unbounded logits, not probabilities: they are
    comparable within one query but NOT across queries.
    """
    cands, timings = retrieve_baseline(query, collection, embedder, TOP_K_RETRIEVE)
    ordered, t_x = rerank(query, cands, cross_encoder)
    timings.update(t_x)
    context = ordered[:TOP_N_CONTEXT]

    t0 = time.perf_counter()
    answer = generate_answer(query, context, model=model)
    timings["generation_s"] = time.perf_counter() - t0
    timings["retrieval_s"] = timings["embed_query_s"] + timings["search_s"]

    return {"pipeline": "reranked", "query": query, "candidates": ordered,
            "context_used": context, "answer": answer, "timings": timings}

In [ ]:
# ---- Smoke test: re-ranking must not BREAK the easy cases ----
_q = "What is the maximum administrative fine under the GDPR?"
_r1, _r2 = run_pipeline_1(_q), run_pipeline_2(_q)

print(f"Q: {_q}\n")
print("top-3 baseline :", [c.chunk_id for c in _r1["context_used"]])
print("top-3 reranked :", [c.chunk_id for c in _r2["context_used"]])
display(show_candidates(_r2["candidates"]))
print("\nANSWER 1 (baseline):\n" + _r1["answer"])
print("\nANSWER 2 (reranked):\n" + _r2["answer"])

for _r in (_r1, _r2):
    assert "20" in _r["answer"], f'{_r["pipeline"]} lost the Art 83(5) amount'
print("\nboth pipelines still answer the easy case correctly")

---
## 6. Failure Mode Analysis (Stress Testing)

Find **at least two** queries where the baseline stumbles and re-ranking recovers. Target these
patterns (the assignment names them explicitly):

| Pattern | What you are engineering |
|---|---|
| **Keyword trap / distractor** | Query shares vocabulary with an irrelevant passage; the real answer uses synonyms |
| **Exception / corner case** | Baseline finds the general rule, misses the "except when…" clause elsewhere |
| **Negation & constraints** | "not allowed", "prohibited", role limits — where embeddings blur positive vs. negative |
| **Buried gold** | The vital chunk sits at dense rank 4–10 and the cross-encoder promotes it into the Top 3 |

**Method:** read your document first and find the clauses that *conflict* or *qualify* each other.
Write queries against those. Queries engineered from the document beat queries invented in the abstract.

In [ ]:
# ---- Failure queries -------------------------------------------------------
# Found by running the hunt below over ~20 candidate queries and keeping the ones where
# the Top-3 changed most. `why_baseline_fails` was written BEFORE looking at the answers.
#
# F4 is deliberately kept even though re-ranking does NOT fix it: a counter-example is
# worth more than four confirmations, and it bounds what the cross-encoder can do.

FAILURE_QUERIES = [
    {
        "id": "F1",
        "pattern": "buried gold / irrelevant top hit",
        "query": "What is the deadline for reporting a personal data breach?",
        "gold_phrase": "not later than 72 hours",
        "gold_source": "Article 33(1)",
        "why_baseline_fails": ("'breach' and 'notification' are spread over Recitals 85-88 "
                               "and Article 70's list of EDPB tasks, which are topically "
                               "similar; the one Article that states the deadline is short "
                               "and reads like any other obligation."),
    },
    {
        "id": "F2",
        "pattern": "recital distractor / binding article buried",
        "query": "Can personal data be transferred abroad without an adequacy decision?",
        "gold_phrase": "In the absence of",
        "gold_source": "Articles 46 and 49",
        "why_baseline_fails": ("Recitals 101-113 discuss international transfers at length "
                               "and in softer wording that matches a natural question more "
                               "closely than the binding Articles do."),
    },
    {
        "id": "F3",
        "pattern": "specific condition buried at rank 10",
        "query": "Can a controller refuse to act on a data subject access request?",
        "gold_phrase": "manifestly unfounded or excessive",
        "gold_source": "Article 12(5)",
        "why_baseline_fails": ("Article 12 is long; the chunk stating the general duty to act "
                               "outranks the chunk holding the exception that permits refusal."),
    },
    {
        "id": "F4",
        "pattern": "keyword trap -- expected to defeat BOTH pipelines",
        "query": "What penalties apply for infringing the Regulation?",
        "gold_phrase": "20 000 000",
        "gold_source": "Article 83(5)",
        "why_baseline_fails": ("Article 84 is literally titled 'Penalties' but contains no "
                               "amounts; the amounts live in Article 83 under the term of art "
                               "'administrative fines'. The query's vocabulary points at the "
                               "decoy."),
    },
]


def find_gold(chunks: list[Chunk], phrase: str) -> list[str]:
    """Chunk ids containing the phrase that must appear for an answer to be correct."""
    return [c.id for c in locate(chunks, phrase)]

In [ ]:
def analyse_query(case: dict, model: str = LLM_MODEL) -> dict:
    """Run both pipelines on one failure case and collect everything the report needs.

    Both pipelines share one retrieval call, so the candidate set is provably identical
    and the only variable is the ordering.
    """
    query = case["query"]
    cands, t = retrieve_baseline(query, collection, embedder, TOP_K_RETRIEVE)
    base_ctx = cands[:TOP_N_CONTEXT]

    t0 = time.perf_counter()
    answer_1 = generate_answer(query, base_ctx, model=model)
    gen1 = time.perf_counter() - t0

    ordered, t_x = rerank(query, cands, cross_encoder)
    re_ctx = ordered[:TOP_N_CONTEXT]

    t0 = time.perf_counter()
    answer_2 = generate_answer(query, re_ctx, model=model)
    gen2 = time.perf_counter() - t0

    gold_ids = set(find_gold(chunks, case["gold_phrase"]))
    in_top3 = lambda ctx: [c.chunk_id for c in ctx if c.chunk_id in gold_ids]
    has = lambda a: case["gold_phrase"].lower() in a.lower()

    return {
        **{k: case[k] for k in ("id", "pattern", "query", "gold_source", "gold_phrase")},
        "candidates": ordered,
        "gold_ids": sorted(gold_ids),
        "gold_in_top3_before": in_top3(base_ctx),
        "gold_in_top3_after": in_top3(re_ctx),
        "top3_before": [c.chunk_id for c in base_ctx],
        "top3_after": [c.chunk_id for c in re_ctx],
        "answer_1": answer_1, "answer_2": answer_2,
        "answer_1_grounded": has(answer_1), "answer_2_grounded": has(answer_2),
        "timings": {**t, **t_x, "gen1_s": gen1, "gen2_s": gen2},
    }


def rank_shift_table(results: list[dict]) -> pd.DataFrame:
    """query | chunk | dense rank -> cross rank | in Top-3 before? | after?

    This table is the core evidence for report section 2.
    """
    rows = []
    for r in results:
        for c in r["candidates"]:
            if c.dense_rank > TOP_N_CONTEXT and c.cross_rank > TOP_N_CONTEXT:
                continue                       # only rows that entered or left the Top-3
            rows.append({
                "case": r["id"], "chunk": c.chunk_id, "label": c.label, "kind": c.kind,
                "dense_rank": c.dense_rank, "cross_rank": c.cross_rank,
                "shift": c.dense_rank - c.cross_rank,
                "top3_before": c.dense_rank <= TOP_N_CONTEXT,
                "top3_after": c.cross_rank <= TOP_N_CONTEXT,
                "is_gold": c.chunk_id in r["gold_ids"],
            })
    return pd.DataFrame(rows)

In [ ]:
failure_results = [analyse_query(case) for case in FAILURE_QUERIES]

# ---- summary: did re-ranking rescue the answer? ----
summary = pd.DataFrame([{
    "case": r["id"], "pattern": r["pattern"][:38], "gold": r["gold_source"],
    "gold_top3_before": bool(r["gold_in_top3_before"]),
    "gold_top3_after": bool(r["gold_in_top3_after"]),
    "A1_grounded": r["answer_1_grounded"], "A2_grounded": r["answer_2_grounded"],
    "top3_changed": len(set(r["top3_before"]) ^ set(r["top3_after"])) // 2,
} for r in failure_results])
display(summary)

print("\n==================== RANK SHIFTS ====================")
display(rank_shift_table(failure_results))

for r in failure_results:
    print("\n" + "=" * 92)
    print(f"{r['id']}  {r['query']}")
    print(f"gold: {r['gold_source']}  (chunks {r['gold_ids']})")
    print(f"top-3 {r['top3_before']}  ->  {r['top3_after']}")
    display(show_candidates(r["candidates"]))
    print(f"\nANSWER 1 (baseline, grounded={r['answer_1_grounded']}):\n{r['answer_1']}")
    print(f"\nANSWER 2 (reranked, grounded={r['answer_2_grounded']}):\n{r['answer_2']}")

---
## 7. Latency Profiling

Separate the stages, or the numbers mean nothing. Report **retrieval** and **re-ranking** apart from
**generation** — LLM time dwarfs both and would hide the effect you are measuring.

In [ ]:
def profile_pipelines(queries: list[str], n_runs: int = 5, warmup: int = 3) -> pd.DataFrame:
    """Time each stage across several queries.

    Stages are reported separately because LLM generation is ~200x everything else and
    would otherwise hide the only cost that differs between the two pipelines.

    The first calls pay model load and lazy init, so `warmup` runs are discarded. On MPS
    the query-embedding time is also noisy, which is why p50 is reported alongside the
    mean -- the mean here is not robust.
    """
    for _ in range(warmup):
        c, _ = retrieve_baseline(queries[0], collection, embedder, TOP_K_RETRIEVE)
        rerank(queries[0], c, cross_encoder)

    acc: dict[str, list[float]] = {"embed_query": [], "vector_search": [], "cross_encode_10": []}
    for _ in range(n_runs):
        for q in queries:
            cands, t = retrieve_baseline(q, collection, embedder, TOP_K_RETRIEVE)
            _, t_x = rerank(q, cands, cross_encoder)
            acc["embed_query"].append(t["embed_query_s"] * 1000)
            acc["vector_search"].append(t["search_s"] * 1000)
            acc["cross_encode_10"].append(t_x["rerank_s"] * 1000)

    rows = []
    for stage, xs in acc.items():
        xs = sorted(xs)
        rows.append({"stage": stage, "n": len(xs), "mean_ms": round(np.mean(xs), 2),
                     "p50_ms": round(xs[len(xs) // 2], 2),
                     "p95_ms": round(xs[int(len(xs) * 0.95)], 2)})
    return pd.DataFrame(rows)


PROBE_QUERIES = [c["query"] for c in FAILURE_QUERIES] + [
    "What are the conditions for valid consent?",
    "Who must appoint a data protection officer?",
    "What is a personal data breach?",
    "When is a DPIA required?",
    "What rights does a data subject have?",
    "What is pseudonymisation?",
]

latency = profile_pipelines(PROBE_QUERIES)
display(latency)

# LLM generation, measured from the failure-case runs above
_gen = [r["timings"][k] * 1000 for r in failure_results for k in ("gen1_s", "gen2_s")]
print(f"llm_generate: n={len(_gen)} mean={np.mean(_gen):.0f}ms p50={np.median(_gen):.0f}ms")
print(f"\none-off indexing cost: {INDEX_EMBED_S:.1f}s for {len(chunks)} chunks")

_p50 = dict(zip(latency["stage"], latency["p50_ms"]))
_base = _p50["embed_query"] + _p50["vector_search"]
print(f"\nretrieval only  : baseline {_base:.1f}ms  ->  reranked "
      f"{_base + _p50['cross_encode_10']:.1f}ms  "
      f"({(_base + _p50['cross_encode_10'])/_base:.1f}x)")
print(f"end-to-end      : re-ranking adds {_p50['cross_encode_10']:.0f}ms to a "
      f"~{np.median(_gen)/1000:.1f}s pipeline "
      f"(+{_p50['cross_encode_10']/np.median(_gen)*100:.1f}%)")

In [ ]:
# ---- Why not cross-encode the whole corpus? Measure it. ----
#
# The bi-encoder embeds each chunk ONCE, offline, and reuses those vectors for every
# query. A cross-encoder cannot: its score depends on the (query, chunk) pair, so nothing
# can be precomputed and the whole cost is paid per query, at query time.

_q = PROBE_QUERIES[0]
_texts = [c.text for c in chunks]
_rows = []
for _n in (10, 25, 50, 100, 250, len(chunks)):
    _pairs = [(_q, t) for t in _texts[:_n]]
    cross_encoder.predict(_pairs[:4], show_progress_bar=False)      # warm
    _t0 = time.perf_counter()
    cross_encoder.predict(_pairs, show_progress_bar=False)
    _el = (time.perf_counter() - _t0) * 1000
    _rows.append({"pairs": _n, "total_ms": round(_el, 1), "ms_per_pair": round(_el / _n, 2)})

scaling_df = pd.DataFrame(_rows)
display(scaling_df)

_full = scaling_df.iloc[-1]["total_ms"]
_search = float(latency.loc[latency["stage"] == "vector_search", "p50_ms"].iloc[0])
print(f"cross-encoding all {len(chunks)} chunks: {_full:.0f}ms per query")
print(f"vector search over the same corpus     : {_search:.2f}ms per query")
print(f"=> {_full/_search:.0f}x slower, on a corpus of only {len(chunks)} chunks")
print("\nAnd the gap widens with scale: cross-encoder cost is linear in corpus size,")
print("while an ANN index is sub-linear. Extrapolating this corpus to 1M chunks:")
print(f"  cross-encoder: ~{_full/len(chunks)*1_000_000/1000/60:.0f} minutes per query")
print("  ANN search   : still single-digit milliseconds")
print("\nHence retrieve-then-rerank: the bi-encoder cheaply narrows 652 -> 10, and the")
print("cross-encoder pays its quadratic-ish attention cost on only those 10.")

---
## 8. Report Notes

The report is submitted alongside this notebook and **must be in your own words** — the assignment
says explicitly that it should not be AI-generated. Use this section only to park the raw numbers and
observations you will write up.

**1. Architecture comparison.** Bi-encoder vs. cross-encoder: where the query and document meet in
each, and what that costs. Explain *why* independent embeddings can blur negation.

**2. Qualitative & rank-shift analysis.** Your table from section 6, plus for each failure query: what
Answer 1 got wrong, and which specific chunk promotion fixed it.

**3. Latency profiling.** Your table from section 7.

**4. Scaling question.** Why not cross-encode the whole collection from the start? Ground it in your
own measured numbers: chunks in your corpus x per-pair cross-encoder cost, and note that chunk
embeddings precompute once while cross-encoder scores cannot be cached per query.

**Checklist before submitting**
- [ ] Single well-commented `.ipynb`, run top to bottom with outputs saved
- [ ] Public/shared GitHub repo, PDF (or its source link) included
- [ ] >= 2 failure queries with documented rank shifts
- [ ] Identical prompt + identical candidate set across both pipelines
- [ ] Latency table
- [ ] Report written by you